# What is actually in these columns

**Module 1 · Session 03, part 1**

The most energetic track in this file is a recording of rain falling on leaves.

That is not a joke about edm. "Rain Forest and Tropical Beach Sound", by an act called Nature
Sounds Nature Music, scores 1.00 on energy, which is the highest value in the dataset. The
least energetic is crickets near a waterfall. Someone put both on a playlist, so Spotify's
audio model measured them exactly the way it measures The Prodigy.

You could have run last week's analysis without ever meeting either of them. Group by genre,
take a mean, write a paragraph about edm, hand it in. The rainforest sits quietly inside the
edm average, and nothing in the pipeline complains.

Today is about the looking that would have caught it.

## The five questions

Work in this order on any table. This notebook goes through them once, on the file you
already know.

| | Question | Where to look |
|---|---|---|
| 1 | How big is it? | `.shape`, `.info()` |
| 2 | What is missing? | `.isna().sum()` |
| 3 | What shape is each column, and is anything impossible? | `.describe()`, then plot it |
| 4 | What groups are there, and how big? | `.value_counts()` |
| 5 | What moves with what? | `.corr()`, plus a picture |

Every one of those is a single line of code, which is why they get skipped.


In [ ]:
# Setup. Same file as last week, plus one style block so every chart here matches.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML

# One theme for the whole notebook, so no chart below needs styling of its own.
# The two colours are chosen to stay distinguishable with colour-blindness.
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8b8a85"
sns.set_theme(style="whitegrid", palette=[BLUE, ORANGE, GREY],
              rc={"figure.dpi": 110, "grid.color": "#ececea", "font.size": 10,
                  "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left"})

# The file lives in the class repository. pandas reads a URL exactly like a file.
URL = "https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/M1_2026/spotify_songs.csv"

songs = pd.read_csv(URL).rename(columns={
    "track_name": "title", "track_artist": "artist",
    "track_popularity": "popularity", "playlist_genre": "genre"})
tracks = songs.drop_duplicates("track_id")   # one row per song instead of one per placement


def listen(rows, *extra_columns):
    """Show rows with a clickable Spotify link. track_id is a real Spotify ID."""
    out = rows[["title", "artist", *extra_columns]].copy()
    out["listen"] = "https://open.spotify.com/track/" + rows["track_id"]
    return HTML(out.to_html(render_links=True, escape=False, index=False))


print(f"Rows (a track on a playlist): {len(songs):,}")
print(f"Distinct songs:               {len(tracks):,}")


Two frames out of one file. `songs` has a row every time a track appears on a playlist, so a
song on five playlists is in there five times. `tracks` has one row per song.

Which one you want depends on what you are claiming. A typical song? Use `tracks`. What is on
these playlists? Use `songs`. Get this wrong and you have not made a coding error, you have
answered a different question.

## 1 and 2. How big, and what is missing


In [ ]:
print("Rows and columns:", songs.shape)

display(songs[["title", "artist", "energy", "danceability", "tempo"]].isna().sum().to_frame("missing"))


Five missing titles and five missing artists, no missing audio. Session 02 settled what to do
about that, which here is nothing: the analysis is about energy, and a row with an unknown
artist still has its energy.

## 3. Is anything impossible?


In [ ]:
display(tracks[["energy", "danceability", "valence", "tempo", "duration_ms"]].describe().round(2))


`describe` gives you eight numbers per column and hides the two things you most need. The
first is the shape.

Each orange line below is the mean of that column. Look at tempo and valence before reading on.


In [ ]:
columns = ["energy", "danceability", "valence", "tempo"]

fig, axes = plt.subplots(2, 2, figsize=(10, 5.5))
for ax, column in zip(axes.flat, columns):
    sns.histplot(tracks, x=column, bins=40, color=BLUE, ax=ax)
    ax.axvline(tracks[column].mean(), color=ORANGE, linewidth=2)   # the mean
    ax.set(title=column, xlabel="", ylabel="songs")
fig.tight_layout()
plt.show()


Four columns, four different situations, one summary statistic.

`danceability` is a single clean hump, and its mean is a fair description of a typical song.

`energy` leans hard towards the top of the range with a long tail to the left, so the mean at
0.70 sits below where most of the songs actually are.

`valence` covers the whole range with a broad plateau in the middle. The mean is 0.51, and
knowing that tells you almost nothing about any particular track.

`tempo` has two clusters, one around 95 beats per minute and one around 125, because produced
music sticks to a small number of conventional tempos. The mean lands on the taller cluster and
the second one disappears from any report that quotes only the average.

The second thing `describe` hides is whether a value is possible at all, and that one you read
straight off the table above.


Four thousand milliseconds is four seconds. A tempo of 0 beats per minute is not a slow song.

Both of those belong to one record, and it is worth playing in class.


In [ ]:
display(listen(tracks.nsmallest(1, "duration_ms"), "duration_ms", "tempo", "valence"))


Four seconds, no tempo, and a valence of 0.0, which makes it simultaneously the shortest, the
least danceable and the saddest thing in the dataset. It is not a sad song. It is not a song.

Nobody suspected that row. It surfaced from the minimum of two columns that have nothing to do
with each other.


In [ ]:
print("Songs under one minute:", int((tracks["duration_ms"] < 60_000).sum()))
print("Songs with tempo exactly 0:", int((tracks["tempo"] == 0).sum()))
print("Out of:", len(tracks))


Twenty-five short ones out of 28,356. A handful, not a crowd, and far too few to move a mean.

Leaving them in and saying you looked is defensible for this file. Quoting a typical song
length in a contract, you would drop them and say so. The rule changes with the claim, which
is why nobody can write it down for you in advance.

### The extremes are where the measurements give themselves away


In [ ]:
extremes = pd.concat([
    tracks.nlargest(1, "energy"), tracks.nsmallest(1, "energy"),
    tracks.nlargest(1, "instrumentalness"), tracks.nlargest(1, "valence"),
    tracks.nlargest(1, "loudness"),
])
display(listen(extremes, "genre", "energy", "valence"))


Top of the energy scale: rainforest. Bottom: crickets. Most instrumental: waves and wind.
Three of the five extreme values in an audio dataset are not music.

Play one. Then look at the word "energy" again. It is a signal-processing measure of loudness,
density and noisiness, and it was never a claim about whether a track is exciting. Write "edm is
the most energetic genre" and you have said something about how the recordings were mastered.

The happiest song in the dataset, at valence 1.00, is "Low Rider" by War, and that one the
model gets right.

## 4. What groups are there, and how big?


In [ ]:
counts = songs["genre"].value_counts()

fig, ax = plt.subplots(figsize=(7, 3.2))
sns.barplot(x=counts.values, y=counts.index, color=BLUE, ax=ax)
ax.bar_label(ax.containers[0], fmt="{:,.0f}".format, padding=4)
ax.set(title="Rows per genre", xlabel="", ylabel="", xlim=(0, counts.max() * 1.15))
ax.set_xticks([])
sns.despine(bottom=True)
plt.show()


Between about 4,900 and 6,000 a side. Unusually even, and not what you normally get.

When it is uneven, a difference between a group of 40 and a group of 40,000 tells you more
about sample sizes than about the world. Print the counts next to every summary. It is a
habit rather than a technique.

## 5. What moves with what?


In [ ]:
correlations = tracks[["energy", "danceability", "loudness", "valence",
                       "acousticness", "popularity"]].corr()
display(correlations["energy"].drop("energy").sort_values(ascending=False).round(3).to_frame("with energy"))


Energy and loudness sit at 0.68. High, and close to a tautology, since both partly measure how
much is going on in the recording.

Energy and danceability sit at -0.08, which is nothing at all. Say that one out loud, because
"energetic" and "danceable" sound like they belong together in English and the data flatly
disagrees.

Picking which of these six numbers deserves a sentence is not something the table does for you.


In [ ]:
sample = tracks.sample(2_000, random_state=2026)   # 28,000 points would be a block of ink

fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.scatterplot(data=sample, x="loudness", y="energy", s=14, alpha=0.3,
                color=BLUE, edgecolor=None, ax=ax)
ax.set(title="Energy against loudness, 2,000 songs sampled with a fixed seed",
       xlabel="loudness (dB)", ylabel="energy", ylim=(0, 1.02))
plt.show()


The cloud is wide. Loudness will not predict energy for any individual song, which the number
0.68 does not tell you on its own.

And there is a tail of very quiet tracks trailing off to the left. Some of those are the
nature recordings from earlier, sitting in a corner of the plot where no music is.

## Your turn

Pick a numeric column nobody has looked at yet. Plot it, say in one sentence whether its mean
is worth reporting, then find the most extreme song in it and play it.


In [ ]:
# One column, one plot, one sentence, one track.


## One answer, using speechiness


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
sns.histplot(tracks, x="speechiness", bins=50, color=BLUE, ax=ax)
ax.axvline(tracks["speechiness"].mean(), color=ORANGE, linewidth=2)
ax.set(title=f"speechiness (orange line = mean, {tracks['speechiness'].mean():.2f})",
       xlabel="speechiness", ylabel="songs")
plt.show()

print("median:", round(tracks["speechiness"].median(), 3))
display(listen(tracks.nlargest(2, "speechiness"), "speechiness", "genre"))


Most songs sit near zero and a thin tail runs right, so the mean lands at 0.11 while the median
is 0.06. Reporting the mean describes almost nobody.

The extreme tracks are mostly talking, which is what the measure is for. Real records, and they
stay. Skew is not error. A long tail is a fact about the world, not a fault in the file, and it
needs a different response from the four-second track earlier.

## Next

Genres differ in energy. Whether any of that difference is worth acting on is a separate
question, and it is the one part 2 spends its time on.
